<a href="https://colab.research.google.com/github/Shubz15/Machine-Learning-Training/blob/main/Exp_10/Experiment_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Part A — Install Required Libraries

In [1]:
# Part A: Install Required Libraries

!pip install nltk scikit-learn pandas

## Part A — Import Libraries

In [2]:
# Part A: Import Libraries

import nltk
import pandas as pd
import string

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

## Part B — Download NLTK Resources

In [3]:
# Part B: Download Required NLTK Data

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

## Part C — Create Sample Dataset

In [4]:
# Part C: Expanded balanced dataset
data = {
    'text': [
        'I love machine learning', 'Python is great', 'I hate bugs', 'Amazing movie', 'Terrible food',
        'I enjoy NLP', 'Bad service', 'AI is fascinating', 'Happy results', 'Horrible experience',
        'Fantastic day', 'I dislike errors', 'Great performance', 'Very disappointing', 'Superb quality',
        'The code is clean', 'I am bored', 'This is the best', 'I am angry', 'Wonderful work',
        'Not good at all', 'I feel successful', 'Waste of time', 'Brilliant idea', 'I am sad',
        'It is failing', 'Worst product ever', 'I am delighted', 'So painful', 'Pure joy',
        'Awful result', 'Really bad', 'Total disaster', 'Extremely poor', 'Not happy'
    ],
    'label': [1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0]
}

df = pd.DataFrame(data)
print(f'Dataset size: {len(df)}')
display(df.tail())

Dataset size: 35


,text,label
30,Awful result,0
31,Really bad,0
32,Total disaster,0
33,Extremely poor,0
34,Not happy,0


## Part D — Text Preprocessing

In [5]:
import nltk
# Part D: Text Preprocessing

import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('punkt_tab', quiet=True)

stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = str(text).lower()
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word.isalpha()]
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [stemmer.stem(word) for word in tokens]
    return " ".join(tokens)

# Applying to the full 35-row dataframe
df['processed_text'] = df['text'].apply(preprocess_text)

print(f'Processed {len(df)} rows.')
print(df[['text', 'processed_text']].tail())

Processed 35 rows.
              text processed_text
30    Awful result      aw result
31      Really bad     realli bad
32  Total disaster   total disast
33  Extremely poor    extrem poor
34       Not happy          happi


## Part E — TF-IDF Vectorization

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer


In [7]:
from sklearn.feature_extraction.text import CountVectorizer
# Part E: Count Vectorization

vectorizer = CountVectorizer()

# Fit on all 35 processed rows
X = vectorizer.fit_transform(df['processed_text'])
y = df['label']

print("Current Vector Shape:", X.shape)

Current Vector Shape: (35, 58)


## Part F — Train-Test Split

In [8]:
# Part F: Train-Test Split

# Split based on all 35 samples
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

Training samples: 28
Testing samples: 7


## Part G — Train Naive Bayes Classifier

In [9]:
from sklearn.ensemble import RandomForestClassifier
# Part G: Re-training Random Forest on the synchronized 35-sample split

model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

print("Random Forest Model successfully retrained on synchronized data.")

Random Forest Model successfully retrained on synchronized data.


## Part H — Prediction

In [10]:
# Part H: Prediction

# Refresh predictions using the updated Random Forest
y_pred = model.predict(X_test)

print(f"Predicted labels: {y_pred}")
print(f"Actual labels:    {y_test.values}")

Predicted labels: [0 0 0 0 0 1 1]
Actual labels:    [1 1 0 0 0 0 1]


## Part I — Evaluation

In [11]:
# Part I: Evaluation
from sklearn.metrics import accuracy_score, classification_report

# Refreshing metrics to reflect the current test set (7 samples)
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, zero_division=0)

print(f"Final Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(report)

Final Accuracy: 57.14%

Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.75      0.67         4
           1       0.50      0.33      0.40         3

    accuracy                           0.57         7
   macro avg       0.55      0.54      0.53         7
weighted avg       0.56      0.57      0.55         7



## Part J — Test New Sentence

In [22]:
# Part J: Test New Sentence

new_text = ["Python is greate"]

# Preprocess
processed = [preprocess_text(text) for text in new_text]

# Convert into TF-IDF
vector = vectorizer.transform(processed)

# Predict
prediction = model.predict(vector)

print("Input Sentence:", new_text[0])

if prediction[0] == 1:
    print("Predicted Sentiment: Positive")
else:
    print("Predicted Sentiment: Negative")

Input Sentence: Python is greate
Predicted Sentiment: Positive


In [23]:
print(f"Processed input: {processed}")
# Check if words from the input are in the vectorizer vocabulary
for word in processed[0].split():
    status = "in" if word in vectorizer.vocabulary_ else "NOT in"
    print(f"Word '{word}' is {status} the vocabulary.")

if 'good' in stop_words:
    print("Note: 'good' is in the stop_words list and was removed.")

Processed input: ['python great']
Word 'python' is in the vocabulary.
Word 'great' is in the vocabulary.
